In [1]:
import geopandas as gpd
import pandas as pd
import folium
import osmnx as ox
from shapely.ops import substring
import networkx as nx
from shapely.geometry import Point, LineString, MultiLineString
from tqdm import tqdm
import numpy as np
from multiprocessing import cpu_count
from multiprocessing import Pool
from functools import partial
import time
import os
from helpers.helpers import process_chunk

In [2]:
ta = gpd.read_file('data/chc-boundaries/territorial-authority-2021-generalised.gpkg', engine='pyogrio')
sa2 = gpd.read_file('data/chc-boundaries/sa2/statistical-area-2-2023-generalised.shp')
canopy = gpd.read_file('data/canopy.gdb')
property_full = gpd.read_file('output/hedonic_gdf.gpkg')

pd.set_option('display.max_columns', None)

ta_chc = ta[ta['TA2021_V1_00_NAME_ASCII'] == 'Christchurch City']
sa2_chc = gpd.clip(sa2, ta_chc)
# boundary = sa2_chc[sa2_chc['SA22023__2'].str.lower().str.contains('ilam')]

property = gpd.clip(property_full, sa2_chc)
TARGET_CRS = 2193

/Users/rujalshrestha/Projects/chc-property-prices/venv/lib/python3.13/site-packages/pyogrio/raw.py:198: RuntimeWarning: organizePolygons() received a polygon with more than 100 parts. The processing may be really slow.  You can skip the processing by setting METHOD=SKIP, or only make it analyze counter-clock wise parts by setting METHOD=ONLY_CCW if you can assume that the outline of holes is counter-clock wise defined
  return ogr_read(


In [3]:
G = ox.graph_from_place(
  'Christchurch, New Zealand',
  network_type='drive',
  simplify=True,
)

G = ox.project_graph(G, to_crs=TARGET_CRS)

edges = ox.graph_to_gdfs(G, nodes=False)
edges = edges.to_crs(property.crs)

# comptute access point

In [4]:
def nearest_street_and_point(point, edges_gdf, max_distance=50):
    distances = edges_gdf.geometry.distance(point)
    min_distance = distances.min()
    
    if min_distance > max_distance:
        return None
    
    idx = distances.idxmin()
    street_geom = edges_gdf.loc[idx].geometry

    proj_dist = street_geom.project(point)
    access_point = street_geom.interpolate(proj_dist)

    return access_point

property["access_point"] = property.geometry.apply(
    lambda p: nearest_street_and_point(p, edges, max_distance=100)
)

property_valid = property[property['access_point'].notna()].copy()
property = property_valid

/Users/rujalshrestha/Projects/chc-property-prices/venv/lib/python3.13/site-packages/geopandas/geodataframe.py:1968: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  super().__setitem__(key, value)


# iso distance calculation

In [ ]:
# MIN_SEGMENT_LEN = 0.5  # metres

# def is_valid_segment(g):
#     return (
#         isinstance(g, LineString)
#         and not g.is_empty
#         and g.length > MIN_SEGMENT_LEN
#     )

# def isodistance_from_access_point(property_row, property_crs, G, G_base, max_dist=200):

#     access_pt = property_row['access_point']
    
#     access_pt_proj = gpd.GeoSeries([access_pt], crs=property_crs).to_crs("EPSG:2193").iloc[0]

#     # Find nearest edge efficiently using spatial index (front street)
#     u, v, k = ox.distance.nearest_edges(G, access_pt_proj.x, access_pt_proj.y, return_dist=False)
    
#     data = G.edges[u, v, k]
    
#     if 'geometry' in data:
#         edge_geom = data['geometry']
#     else:
#         # Create geometry from node coordinates
#         u_pt = Point(G.nodes[u]['x'], G.nodes[u]['y'])
#         v_pt = Point(G.nodes[v]['x'], G.nodes[v]['y'])
#         edge_geom = LineString([u_pt, v_pt])
        
#     edge_length = data['length']
    
#     proj_dist = edge_geom.project(access_pt_proj)
#     proj_dist = min(max(proj_dist, 0), edge_length)

#     dist_to_u = proj_dist
#     dist_to_v = edge_length - proj_dist
    
#     needs_split = not (dist_to_u < 0.5 or dist_to_v < 0.5)

#     if needs_split:
#         G_work = G_base.copy()
#     else:
#         G_work = G_base

#     if dist_to_u < 0.5:
#         start_node = u
        
#     elif dist_to_v < 0.5:
#         start_node = v
        
#     else:
#         access_node = f"access_{property_row.name}"
#         G_work.add_node(access_node, x=access_pt_proj.x, y=access_pt_proj.y)

#         geom_to_u = substring(edge_geom, 0, proj_dist)
#         geom_to_v = substring(edge_geom, proj_dist, edge_length)

#         G_work.add_edge(access_node, u, length=dist_to_u, geometry=geom_to_u)
#         G_work.add_edge(access_node, v, length=dist_to_v, geometry=geom_to_v)

#         if G_work.has_edge(u, v):
#             G_work.remove_edge(u, v, k)
            
#         start_node = access_node
    
#     lengths = nx.single_source_dijkstra_path_length(
#         G_work,
#         start_node,
#         cutoff=max_dist,
#         weight='length'
#     )

#     reachable = set(lengths.keys())

#     lines = []

#     for u_node, v_node, data in G_work.edges(data=True):
        
#         if u_node not in reachable and v_node not in reachable:
#             continue
        
#         du = lengths.get(u_node, float('inf'))
#         dv = lengths.get(v_node, float('inf'))
        
#         if du > max_dist and dv > max_dist:
#             continue
                
#         geom = data.get('geometry')
        
#         if geom is None:
#             u_pt = Point(G_work.nodes[u_node]['x'], G_work.nodes[u_node]['y'])
#             v_pt = Point(G_work.nodes[v_node]['x'], G_work.nodes[v_node]['y'])
#             geom = LineString([u_pt, v_pt])
            
#         edge_len = data.get('length', geom.length)

#         if du <= max_dist and dv <= max_dist:
#             lines.append(geom)

#         elif du <= max_dist < dv:
#             remaining = max_dist - du
#             if remaining > MIN_SEGMENT_LEN:
#                 u_pt = Point(G_work.nodes[u_node]['x'], G_work.nodes[u_node]['y'])
#                 v_pt = Point(G_work.nodes[v_node]['x'], G_work.nodes[v_node]['y'])
                
#                 geom_start = Point(geom.coords[0])
                
#                 if geom_start.distance(u_pt) < geom_start.distance(v_pt):
#                     clipped = substring(geom, 0, min(remaining, edge_len))
#                 else:
#                     start_dist = max(0, edge_len - remaining)
#                     clipped = substring(geom, start_dist, edge_len)
                    
#                 if is_valid_segment(clipped):
#                     lines.append(clipped)

#         elif dv <= max_dist < du:
#             remaining = max_dist - dv
#             if remaining > MIN_SEGMENT_LEN:
#                 u_pt = Point(G_work.nodes[u_node]['x'], G_work.nodes[u_node]['y'])
#                 v_pt = Point(G_work.nodes[v_node]['x'], G_work.nodes[v_node]['y'])
                
#                 geom_start = Point(geom.coords[0])
                
#                 if geom_start.distance(v_pt) < geom_start.distance(u_pt):
#                     clipped = substring(geom, 0, min(remaining, edge_len))
#                 else:
#                     start_dist = max(0, edge_len - remaining)
#                     clipped = substring(geom, start_dist, edge_len)
                    
#                 if is_valid_segment(clipped):
#                     lines.append(clipped)

#     if not lines:
#         return MultiLineString([])

#     result = MultiLineString(lines)
#     return gpd.GeoSeries([result], crs=G.graph['crs']).to_crs(property_crs).iloc[0]

# for all properties

In [ ]:
# def compute_reach_and_buffer(row, property_crs, G, G_base, max_dist=200, buffer_dist=10):
#     try:
#         reach = isodistance_from_access_point(
#             row,
#             property_crs,
#             G,
#             G_base,
#             max_dist=max_dist
#         )

#         if reach is None or reach.is_empty:
#             return MultiLineString([]), None

#         reach_buffer = reach.buffer(buffer_dist)

#         if reach_buffer.is_empty:
#             return reach, reach_buffer

#         return reach, reach_buffer

#     except Exception as e:
#         print(f"Error for property {row.name}: {e}")
#         return MultiLineString([]), None

# for one property testing

In [ ]:
# # 3247, 2451, 4765
# single_prop = property.loc[3247]

# edges_wgs = edges.to_crs(4326)
# nodes = ox.graph_to_gdfs(G, nodes=True, edges=False)
# nodes_wgs = nodes.to_crs(4326)

# G_base = G.to_undirected(as_view=False)
# result = compute_reach_and_buffer(single_prop, property.crs, G, G_base)

# reach_200m, reach_buffer = result

# # reproject to WGS84 for folium
# wgs = gpd.GeoSeries(
#     [single_prop.geometry, single_prop.access_point],
#     crs=property.crs
# ).to_crs(4326)

# [prop_wgs, access_wgs] = wgs

# reach_wgs = (
#     gpd.GeoSeries([reach_200m], crs=property.crs)
#     .to_crs(4326)
#     .iloc[0]
#     if reach_200m is not None else None
# )

# buffer_wgs = (
#     gpd.GeoSeries([reach_buffer], crs=property.crs)
#     .to_crs(4326)
#     .iloc[0]
#     if reach_buffer is not None else None
# )

# # map
# m = folium.Map(
#     location=[prop_wgs.y, prop_wgs.x],
#     zoom_start=16,
#     tiles="CartoDB positron"
# )

# # property point
# folium.CircleMarker(
#     [prop_wgs.y, prop_wgs.x],
#     radius=7,
#     color="purple",
#     fill=True,
#     fill_opacity=1,
#     tooltip=f"Property index: {single_prop.name}"
# ).add_to(m)

# folium.CircleMarker(
#     [access_wgs.y, access_wgs.x],
#     radius=4,
#     color="brown",
#     fill=True,
#     fill_opacity=1,
# ).add_to(m)

# # reachable streets
# if reach_wgs is not None and not reach_wgs.is_empty:
#     folium.GeoJson(
#         reach_wgs,
#         style_function=lambda x: {
#             "color": "red",
#             "weight": 3,
#             "opacity": 0.7
#         },
#         tooltip="Reachable streets (200m)"
#     ).add_to(m)

# # buffer
# if buffer_wgs is not None and not buffer_wgs.is_empty:
#     folium.GeoJson(
#         buffer_wgs,
#         style_function=lambda x: {
#             "color": "orange",
#             "fillColor": "orange",
#             "weight": 1,
#             "fillOpacity": 0.25
#         },
#         tooltip="Street buffer"
#     ).add_to(m)

# folium.GeoJson(
#     edges_wgs,
#     name="Street network",
#     style_function=lambda x: {
#         "color": "#555555",
#         "weight": 1,
#         "opacity": 0.5
        
#     }
# ).add_to(m)

# folium.GeoJson(
#     nodes_wgs,
#     name="Street nodes",
#     marker=folium.CircleMarker(
#         radius=1.5,
#         color="#1f78b4",
#         fill=True,
        
#         fill_opacity=0.6
#     ),
#     tooltip=folium.GeoJsonTooltip(
#         fields=[],
#         aliases=[]
#     )
# ).add_to(m)

# m

# for sample properties

In [ ]:
# tqdm.pandas(desc="Computing isodistance")

# G_base = G.to_undirected()

# sub_property = property.sample(n=100, random_state=7).copy()

# results = sub_property.progress_apply(
#     lambda row: compute_reach_and_buffer(
#         row,
#         sub_property.crs,
#         G,
#         G_base,
#         max_dist=300,
#         buffer_dist=10
#     ),
#     axis=1
# )

# sub_property['reach_200m'] = results.apply(lambda x: x[0])
# sub_property['reach_buffer'] = results.apply(lambda x: x[1]) 

Computing isodistance: 100%|██████████| 100/100 [00:18<00:00,  5.42it/s]


# plotting

In [ ]:
# # canopy_wgs = canopy.to_crs(4326)
# props_wgs = sub_property.geometry.to_crs(4326)

# m = folium.Map(
#     location=[props_wgs.y.mean(), props_wgs.x.mean()],
#     zoom_start=15,
#     tiles="CartoDB positron"
# )

# for idx, row in sub_property.iterrows():

#     # ---- reproject core geometries ----
#     prop = gpd.GeoSeries([row.geometry], crs=property.crs).to_crs(4326).iloc[0]
#     access = gpd.GeoSeries([row['access_point']], crs=property.crs).to_crs(4326).iloc[0]
#     reach = gpd.GeoSeries([row['reach_200m']], crs=property.crs).to_crs(4326).iloc[0]
#     buffer_geom = gpd.GeoSeries([row['reach_buffer']], crs=property.crs).to_crs(4326).iloc[0]

#     # ---- property ----
#     folium.CircleMarker(
#         [prop.y, prop.x],
#         radius=7,
#         color="purple",
#         fill=True,
#         fill_opacity=1,
#         popup=f"Property {idx}"
#     ).add_to(m)

#     # ---- access point ----
#     folium.CircleMarker(
#         [access.y, access.x],
#         radius=4,
#         color="blue",
#         fill=True,
#         fill_opacity=0.9,
#         popup=f"Access {idx}"
#     ).add_to(m)
    
#     # ---- line from property to access point ----
#     folium.PolyLine(
#         locations=[
#             [prop.y, prop.x],
#             [access.y, access.x]
#         ],
#         color="purple",
#         weight=2,
#         opacity=0.7,
#         dash_array="6, 6",
#         tooltip=f"Property → Access ({idx})"
#     ).add_to(m)

#     # ---- reachable streets ----
#     if not reach.is_empty:
#         folium.GeoJson(
#             reach,
#             style_function=lambda x: {
#                 "color": "red",
#                 "weight": 3,
#                 "opacity": 0.6
#             },
#             tooltip=f"Reachable streets ({idx})"
#         ).add_to(m)

#     # ---- street buffer ----
#     if buffer_geom is not None and (not buffer_geom.is_empty):
#         folium.GeoJson(
#             buffer_geom,
#             style_function=lambda x: {
#                 "color": "orange",
#                 "fillColor": "orange",
#                 "weight": 1,
#                 "fillOpacity": 0.25
#             },
#             tooltip=f"5m street buffer ({idx})"
#         ).add_to(m)

#     # ---- canopy intersecting buffer ----
#     # if buffer_geom is not None and (not buffer_geom.is_empty):
#     #     canopy_clip = canopy_wgs[canopy_wgs.intersects(buffer_geom)]

#     #     if not canopy_clip.empty:
#     #         folium.GeoJson(
#     #             canopy_clip,
#     #             style_function=lambda x: {
#     #                 "color": "darkgreen",
#     #                 "fillColor": "darkgreen",
#     #                 "weight": 1,
#     #                 "fillOpacity": 0.6
#     #             },
#     #             tooltip=f"Canopy in buffer ({idx})"
#     #         ).add_to(m)

# m

# applying for all properties

In [11]:
# tqdm.pandas(desc="Computing isodistance")

# G_base = G.to_undirected()

# sub_property = property.sample(n=1000, random_state=4).copy()

# results = sub_property.progress_apply(
#     lambda row: compute_reach_and_buffer(
#         row,
#         sub_property.crs,
#         G,
#         G_base,
#         max_dist=200,
#         buffer_dist=10
#     ),
#     axis=1
# )

# sub_property['reach_200m'] = results.apply(lambda x: x[0])
# sub_property['reach_buffer'] = results.apply(lambda x: x[1]) 

# using parallel processing

In [13]:
ISO_LENGTH_REACH = 150

In [14]:
n_cores = cpu_count()
n_chunks = n_cores * 2

chunks = np.array_split(property, n_chunks)

G_base = G.to_undirected(as_view=False)

if __name__ == "__main__":
    with Pool(processes=n_cores) as pool:

        func = partial(
            process_chunk,
            crs=property.crs,
            G=G,
            G_base=G_base,
            max_dist=ISO_LENGTH_REACH,
            buffer_dist=10
        )

        results = list(
            tqdm(
                pool.imap(func, chunks),
                total=len(chunks),
                desc="Parallel isodistance"
            )
        )


/Users/rujalshrestha/Projects/chc-property-prices/venv/lib/python3.13/site-packages/numpy/_core/fromnumeric.py:57: FutureWarning: 'GeoDataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'GeoDataFrame.transpose' instead.
  return bound(*args, **kwds)
/Users/rujalshrestha/Projects/chc-property-prices/venv/lib/python3.13/site-packages/numpy/_core/fromnumeric.py:57: FutureWarning: 'GeoDataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'GeoDataFrame.transpose' instead.
  return bound(*args, **kwds)
/Users/rujalshrestha/Projects/chc-property-prices/venv/lib/python3.13/site-packages/numpy/_core/fromnumeric.py:57: FutureWarning: 'GeoDataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'GeoDataFrame.transpose' instead.
  return bound(*args, **kwds)
/Users/rujalshrestha/Projects/chc-property-prices/venv/lib/python3.13/site-packages/numpy/_core/fromnumeric.py:57: FutureWarning: 'GeoDataFrame.swap

[PID 27126] 10/616 rows | 3.1s elapsed | 3.23 rows/s
[PID 27125] 10/616 rows | 3.1s elapsed | 3.19 rows/s
[PID 27124] 10/616 rows | 3.2s elapsed | 3.08 rows/s
[PID 27127] 10/616 rows | 3.0s elapsed | 3.33 rows/s
[PID 27128] 10/616 rows | 3.1s elapsed | 3.18 rows/s
[PID 27129] 10/616 rows | 3.0s elapsed | 3.31 rows/s
[PID 27130] 10/616 rows | 3.0s elapsed | 3.36 rows/s
[PID 27131] 10/616 rows | 2.9s elapsed | 3.41 rows/s
[PID 27132] 10/616 rows | 2.8s elapsed | 3.58 rows/s
[PID 27133] 10/616 rows | 2.6s elapsed | 3.78 rows/s
[PID 27125] 20/616 rows | 6.3s elapsed | 3.17 rows/s
[PID 27126] 20/616 rows | 6.4s elapsed | 3.13 rows/s
[PID 27124] 20/616 rows | 6.8s elapsed | 2.93 rows/s
[PID 27127] 20/616 rows | 6.6s elapsed | 3.03 rows/s
[PID 27128] 20/616 rows | 6.5s elapsed | 3.10 rows/s
[PID 27130] 20/616 rows | 6.2s elapsed | 3.25 rows/s
[PID 27129] 20/616 rows | 6.6s elapsed | 3.02 rows/s
[PID 27131] 20/616 rows | 6.4s elapsed | 3.11 rows/s
[PID 27132] 20/616 rows | 6.4s elapsed | 3.15 

Parallel isodistance:   5%|▌         | 1/20 [03:37<1:09:00, 217.93s/it]

[PID 27125] chunk done in 215.4s
[PID 27132] chunk done in 213.4s
[PID 27133] chunk done in 213.5s
[PID 27131] 10/616 rows | 3.5s elapsed | 2.87 rows/s
[PID 27129] 610/616 rows | 215.4s elapsed | 2.83 rows/s
[PID 27126] 10/616 rows | 3.4s elapsed | 2.98 rows/s
[PID 27124] 10/616 rows | 3.5s elapsed | 2.86 rows/s


Parallel isodistance:  20%|██        | 4/20 [03:40<11:09, 41.82s/it]   

...

[PID 27125] 610/616 rows | 218.2s elapsed | 2.80 rows/s
[PID 27126] chunk done in 220.8s
[PID 27127] chunk done in 216.5s
[PID 27133] 610/615 rows | 218.2s elapsed | 2.80 rows/s
[PID 27128] 610/616 rows | 219.6s elapsed | 2.78 rows/s
[PID 27129] 600/615 rows | 215.7s elapsed | 2.78 rows/s
[PID 27132] 610/616 rows | 219.6s elapsed | 2.78 rows/s
[PID 27125] chunk done in 220.1s
[PID 27130] 610/616 rows | 222.0s elapsed | 2.75 rows/s
[PID 27133] chunk done in 219.8s
[PID 27128] chunk done in 221.4s
[PID 27132] chunk done in 221.1s


Parallel isodistance:  70%|███████   | 14/20 [07:20<02:10, 21.82s/it]

[PID 27129] 610/615 rows | 218.1s elapsed | 2.80 rows/s
[PID 27130] chunk done in 223.6s


Parallel isodistance: 100%|██████████| 20/20 [07:21<00:00, 22.06s/it]

[PID 27129] chunk done in 219.2s


In [15]:
reach_col = f"reach_{ISO_LENGTH_REACH}m"

property[reach_col] = None

for chunk_result in results:
    for idx, reach, buffer_geom in chunk_result:
        property.at[idx, reach_col] = reach

# export reach geometry

In [16]:
property = property.reset_index(drop=True)
property['property_id'] = property.index.astype(int)

reach_networks = gpd.GeoDataFrame(
    property[['property_id', reach_col]],
    geometry=reach_col,
    crs=property.crs
)
reach_networks = reach_networks[
  reach_networks[reach_col].notna() &
  ~reach_networks[reach_col].is_empty
]
reach_networks.to_file(f'output/property_reach_{ISO_LENGTH_REACH}m.gpkg', driver='GPKG', engine='pyogrio')

# export access points

In [ ]:
access_points = gpd.GeoDataFrame(
    property[['property_id', 'access_point']],
    geometry='access_point',
    crs=property.crs
)
access_points.to_file('output/property_accesspoints.gpkg', driver='GPKG', engine='pyogrio')

In [8]:
property.columns

Index(['GrossSalePrice', 'AgeAtSale', 'LandArea', 'TotalFloorArea',
       'water_DIST', 'bus_DIST', 'Census_Pop', 'RnkIMDNoEm', 'RnkIMDNoIn',
       'RnkIMDNoCr', 'RnkIMDNoHo', 'RnkIMDNoHe', 'RnkIMDNoEd', 'RnkIMDNoAc',
       'DECILE_high', 'DECILE_prim', 'Median_Income', 'CBD_DIST',
       'cycleways_DIST', 'cycle_DENS', 'year_2018', 'year_2019', 'canopy_0_20',
       'canopy_20_50', 'canopy_50_100', 'canopy_100_200', 'residuals',
       'geometry', 'access_point', 'reach_200m', 'reach_buffer',
       'property_id'],
      dtype='object')

In [9]:
property.drop(columns=['access_point', 'reach_200m', 'reach_buffer']).to_file('output/property.gpkg', driver='GPKG', engine='pyogrio')